In [1]:
%matplotlib inline

from collections import OrderedDict
from importlib import reload
import os, sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

import numpy as np
import pandas as pd
import scipy.stats

from astropy.table import Table
from astropy.stats import mad_std
from astropy.constants import iau2012 as const

from scipy.stats import spearmanr, gaussian_kde
import statsmodels.api as sm

sys.path.append('../aklpyutils/')
sys.path.append('../')
from utils_stats import *

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MaxNLocator
from shapely.geometry import Point
from shapely.ops import cascaded_union

from jsplottools.sun_plot_tools.general import mpl_setup
from jsplottools.sun_plot_tools.ax import *
from jsplottools.sun_plot_tools.fig import *
mpl_setup(figtype='paper-1/3')
params = {'legend.fontsize': 'x-large',          
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
plt.rcParams.update(params)

In [2]:
from key_dirs import *
from utils_spacing import *

['ic1954', 'ic5273', 'ngc0253', 'ngc0300', 'ngc0628', 'ngc0685', 'ngc1097', 'ngc1511', 'ngc1546', 'ngc1637', 'ngc1792', 'ngc2903', 'ngc2997', 'ngc3137', 'ngc3351', 'ngc3489', 'ngc3511', 'ngc3521', 'ngc3621', 'ngc3627', 'ngc4254', 'ngc4298', 'ngc4459', 'ngc4476', 'ngc4477', 'ngc4536', 'ngc4548', 'ngc4569', 'ngc4596', 'ngc4731', 'ngc4781', 'ngc4826', 'ngc4941', 'ngc5068', 'ngc5128', 'ngc5236', 'ngc5643', 'ngc7456', 'ngc7496', 'ngc7793']


In [3]:
nthresh = 50

**Load the Correlation Functions**

In [4]:
corr_dict = {}
first = True
for this_source in source_list:
    
    # Load files
    
    expvsflat_file = corr_dir + this_source+'_150pc_expvsflat.fits'
    realvsflat_file = corr_dir + this_source+'_150pc_realvsflat.fits'
    realvsexp_file = corr_dir + this_source+'_150pc_realvsexp.fits'
    
    if os.path.isfile(realvsflat_file) == False:        
        print(this_source, " missing a correlation file.")
        continue     
    
    realvsexp_tab = Table.read(realvsexp_file)
    realvsflat_tab = Table.read(realvsflat_file)
    expvsflat_tab = Table.read(expvsflat_file)
    
    fidlogx = 3.0
    
    templogx = np.log10(realvsflat_tab['xgeo'])
    templogy = np.log10(realvsflat_tab['twopt'])
    realvsflat_atfid = np.interp(fidlogx,templogx,templogy,left=np.nan,right=np.nan)

    templogx = np.log10(realvsexp_tab['xgeo'])
    templogy = np.log10(realvsexp_tab['twopt'])
    realvsexp_atfid = np.interp(fidlogx,templogx,templogy,left=np.nan,right=np.nan)

    templogx = np.log10(expvsflat_tab['xgeo'])
    templogy = np.log10(expvsflat_tab['twopt'])
    expvsflat_atfid = np.interp(fidlogx,templogx,templogy,left=np.nan,right=np.nan)

    fid_cat = Table.read(fid_cat_dict[this_source].replace('_props.fits','_props_valueadded.fits'))
    max_rad_deg = np.nanmax(np.sqrt(fid_cat['DELTAX_DEG']**2+fid_cat['DELTAY_DEG']**2))
    
    # Store some arrays if first one
    if first:        
        bins_xlo = np.array(realvsflat_tab['xlo'])
        bins_xhi = np.array(realvsflat_tab['xhi'])
        bins_xgeo = np.array(realvsflat_tab['xgeo'])
        bins_xmid = np.array(realvsflat_tab['xmid'])
        nbins = len(bins_xgeo)        
        first=False
    
    corr_dict[this_source] = {
        'nclouds':len(fid_cat),
        'max_rad_deg':max_rad_deg,
        'fidlogx':fidlogx,
        'realvsexp':realvsexp_tab,
        'realvsexp_atfid':realvsexp_atfid,
        'realvsflat':realvsflat_tab,
        'realvsflat_atfid':realvsflat_atfid,
        'expvsflat':expvsflat_tab,
        'expvsflat_atfid':expvsflat_atfid,
    }
    
    #print(this_source, corr_dict[this_source]['nclouds'])

ic1954  missing a correlation file.
ic5273  missing a correlation file.
ngc0253  missing a correlation file.
ngc0300  missing a correlation file.
ngc0628  missing a correlation file.
ngc0685  missing a correlation file.
ngc1097  missing a correlation file.
ngc1511  missing a correlation file.
ngc1546  missing a correlation file.
ngc1637  missing a correlation file.
ngc1792  missing a correlation file.
ngc2903  missing a correlation file.
ngc2997  missing a correlation file.
ngc3137  missing a correlation file.
ngc3351  missing a correlation file.
ngc3489  missing a correlation file.
ngc3511  missing a correlation file.
ngc3521  missing a correlation file.
ngc3621  missing a correlation file.
ngc3627  missing a correlation file.
ngc4254  missing a correlation file.
ngc4298  missing a correlation file.
ngc4459  missing a correlation file.
ngc4476  missing a correlation file.
ngc4477  missing a correlation file.
ngc4536  missing a correlation file.
ngc4548  missing a correlation file.
ngc

**Load some galaxy properties**

In [5]:
source_list_ra = np.array(source_list)

mass_ra = np.zeros(len(source_list_ra))
lco_ra = np.zeros(len(source_list_ra))
sfr_ra = np.zeros(len(source_list_ra))
lstar_ra = np.zeros(len(source_list_ra))
incl_ra = np.zeros(len(source_list_ra))

ncloud_ra = np.zeros(len(source_list_ra))*np.nan
norm_realflat_ra = np.zeros(len(source_list_ra))*np.nan
norm_realexp_ra = np.zeros(len(source_list_ra))*np.nan
norm_expflat_ra = np.zeros(len(source_list_ra))*np.nan
max_rad_ra = np.zeros(len(source_list_ra))*np.nan

for ii, this_source in enumerate(source_list_ra):
    deets = deets_dict[this_source]
    mass_ra[ii] = float(deets['props_mstar'])
    sfr_ra[ii] = float(deets['props_sfr'])
    lco_ra[ii] = float(deets['lco_phangs'])
    incl_ra[ii] = float(deets['orient_incl'])
    deg_to_pc = float(np.pi/180.*deets['dist']*1e3)
    lstar_ra[ii] = float(deets['size_scalelength']/3600.*deg_to_pc)
    if this_source in corr_dict.keys():
        ncloud_ra[ii] = corr_dict[this_source]['nclouds']
        norm_realflat_ra[ii] = corr_dict[this_source]['realvsflat_atfid']
        norm_realexp_ra[ii] = corr_dict[this_source]['realvsexp_atfid']
        norm_expflat_ra[ii] = corr_dict[this_source]['expvsflat_atfid']
        max_rad_ra[ii] = corr_dict[this_source]['max_rad_deg']*deg_to_pc
    if norm_realflat_ra[ii] == 1.:
        print(this_source)
        
print(norm_realflat_ra)
ndens_ra = ncloud_ra*1./np.pi/lstar_ra**2

mass_order = np.argsort(mass_ra)
lco_order = np.argsort(lco_ra)
sfr_order = np.argsort(sfr_ra)
lstar_order = np.argsort(lstar_ra)
ncloud_order = np.argsort(ncloud_ra)
ndens_order = np.argsort(ndens_ra)

print("Total galaxies above threshold: ", np.sum(ncloud_ra > nthresh), " below ", np.sum(ncloud_ra <= nthresh))

C:\Users\haohe\AppData\Local\Temp\ipykernel_27804\3720637097.py:19: UserWarning: Warning: converting a masked element to nan.
  lco_ra[ii] = float(deets['lco_phangs'])


[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan]
Total galaxies above threshold:  0  below  0


**Bin all galaxies into aggregated trends**

In [6]:
# Initialize output

realvsflat_16 = np.zeros_like(bins_xgeo)*np.nan
realvsflat_50 = np.zeros_like(bins_xgeo)*np.nan
realvsflat_84 = np.zeros_like(bins_xgeo)*np.nan
realvsflat_errlo = np.zeros_like(bins_xgeo)*np.nan
realvsflat_errhi = np.zeros_like(bins_xgeo)*np.nan

realvsexp_16 = np.zeros_like(bins_xgeo)*np.nan
realvsexp_50 = np.zeros_like(bins_xgeo)*np.nan
realvsexp_84 = np.zeros_like(bins_xgeo)*np.nan
realvsexp_errlo = np.zeros_like(bins_xgeo)*np.nan
realvsexp_errhi = np.zeros_like(bins_xgeo)*np.nan

expvsflat_16 = np.zeros_like(bins_xgeo)*np.nan
expvsflat_50 = np.zeros_like(bins_xgeo)*np.nan
expvsflat_84 = np.zeros_like(bins_xgeo)*np.nan
expvsflat_errlo = np.zeros_like(bins_xgeo)*np.nan
expvsflat_errhi = np.zeros_like(bins_xgeo)*np.nan

# Loop over the bins

for ii in range(nbins):

    # Extract vectors for this bin
    realvsflat_list = []
    realvsexp_list = []
    expvsflat_list = []
    
    for this_source in corr_dict.keys():
        this_entry = corr_dict[this_source]
        realvsflat_list.append(np.array(this_entry['realvsflat']['twopt'])[ii])
        realvsexp_list.append(np.array(this_entry['realvsexp']['twopt'])[ii])
        expvsflat_list.append(np.array(this_entry['expvsflat']['twopt'])[ii])
        
    realvsflat_ra = np.array(realvsflat_list)
    realvsexp_ra = np.array(realvsexp_list)
    expvsflat_ra = np.array(expvsflat_list)   
    
    # Caclulate perecentiles and save results
    
    vec = realvsflat_ra
    realvsflat_16[ii] = np.nanpercentile(vec,16)
    realvsflat_50[ii] = np.nanpercentile(vec,50)
    realvsflat_84[ii] = np.nanpercentile(vec,84)
    n_fin = np.sum(np.isfinite(vec))
    realvsflat_errlo[ii] = \
        (np.nanpercentile(vec,50) - np.nanpercentile(vec,16))/np.sqrt(n_fin)
    realvsflat_errhi [ii]= \
        (np.nanpercentile(vec,84) - np.nanpercentile(vec,50))/np.sqrt(n_fin)

    vec = realvsexp_ra
    realvsexp_16[ii] = np.nanpercentile(vec,16)
    realvsexp_50[ii] = np.nanpercentile(vec,50)
    realvsexp_84[ii] = np.nanpercentile(vec,84)
    n_fin = np.sum(np.isfinite(vec))
    realvsexp_errlo[ii] = \
        (np.nanpercentile(vec,50) - np.nanpercentile(vec,16))/np.sqrt(n_fin)
    realvsexp_errhi [ii]= \
        (np.nanpercentile(vec,84) - np.nanpercentile(vec,50))/np.sqrt(n_fin)

    vec = expvsflat_ra
    expvsflat_16[ii] = np.nanpercentile(vec,16)
    expvsflat_50[ii] = np.nanpercentile(vec,50)
    expvsflat_84[ii] = np.nanpercentile(vec,84)
    n_fin = np.sum(np.isfinite(vec))
    expvsflat_errlo[ii] = \
        (np.nanpercentile(vec,50) - np.nanpercentile(vec,16))/np.sqrt(n_fin)*np.sqrt(np.pi/2.0)
    expvsflat_errhi [ii]= \
        (np.nanpercentile(vec,84) - np.nanpercentile(vec,50))/np.sqrt(n_fin)*np.sqrt(np.pi/2.0)

NameError: name 'bins_xgeo' is not defined

In [ ]:
ind = np.isfinite(ndens_ra)

print("Correlation with ncloud")
print(spearmanr(ncloud_ra[ind],norm_realflat_ra[ind]))
print(spearmanr(ncloud_ra[ind],norm_realexp_ra[ind]))
print(spearmanr(ncloud_ra[ind],norm_expflat_ra[ind]))
print("")
print("Correlation with lstar")
print(spearmanr(lstar_ra[ind],norm_realflat_ra[ind]))
print(spearmanr(lstar_ra[ind],norm_realexp_ra[ind]))
print(spearmanr(lstar_ra[ind],norm_expflat_ra[ind]))
print("")
print("Correlation with number density")
print(spearmanr(ndens_ra[ind],norm_realflat_ra[ind]))
print(spearmanr(ndens_ra[ind],norm_realexp_ra[ind]))
print(spearmanr(ndens_ra[ind],norm_expflat_ra[ind]))
print("")
print("Correlation with max radius in units of lstar")
print(spearmanr(max_rad_ra[ind]/lstar_ra[ind],norm_realflat_ra[ind]))
print(spearmanr(max_rad_ra[ind]/lstar_ra[ind],norm_realexp_ra[ind]))
print(spearmanr(max_rad_ra[ind]/lstar_ra[ind],norm_expflat_ra[ind]))

plt.scatter(ndens_ra[ind],norm_expflat_ra[ind],marker='o',s=40)

In [ ]:
png_name = '../plots/twopt_multipanel.png'

npanels = 6
ncol = 2
nrow = 3
ticklength = 5

xmin = +1.75
xmax = +4.0
ymin = -0.5
ymax = +1.0
xticks = np.arange(xmin,xmax,0.5)
yticks = np.arange(ymin,ymax,0.5)

mpl_setup('paper-1/1', aspect=3./3., fs=12)

fig, axes = plt.subplots(
    nrow, ncol, sharex='all', sharey='all',
    gridspec_kw={'wspace': 0.1, 'hspace': 0.1})

axes = axes.ravel()

xlabel = r'$\log_{10}$ Separation [pc]'
ylabel = r'$\log_{10} 1 + \omega$'

fig.text(0.015, 0.5, ylabel, va='center', ha='center', rotation='vertical', fontsize='large')
fig.text(0.5, 0.015, xlabel, va='center', ha='center', rotation='horizontal', fontsize='large')

rows_done = 0
counter = 0

beam_scale = np.log10(150.)

for iax, ax in enumerate(axes):
    
    axes[iax].set_xlim([xmin,xmax])
    axes[iax].set_xscale('linear')
    axes[iax].set_ylim([ymin,ymax])
    axes[iax].set_xticks(xticks)
    axes[iax].set_yticks(yticks)
    axes[iax].tick_params(direction='inout', length=ticklength)
       
    axes[iax].grid(True, linestyle='dotted',linewidth=0.5,color='black')

    plot_main_legend = False
    plot_bbox = False
    plot_aggregate = False
    
    if counter == 0:
        this_label = r'\textbf{Data vs. Flat}'
        plot_bbox = True
        this_tab_tag = 'realvsflat'        
        
    if counter == 1:
        this_label = r'\textbf{Data vs. Flat}'
        this_tab_tag = 'realvsflat'
        plot_aggregate = True        
        
        agg_16 = realvsflat_16
        agg_50 = realvsflat_50
        agg_84 = realvsflat_84
        agg_lo = realvsflat_errlo
        agg_hi = realvsflat_errhi        

    if counter == 2:
        this_label = r'\textbf{Exponential vs. Flat}'
        this_tab_tag = 'expvsflat'        
        plot_bbox = True
        
    if counter == 3:
        this_label = r'\textbf{Exponential vs. Flat}'
        this_tab_tag = 'expvsflat'
        plot_aggregate = True        

        agg_16 = expvsflat_16
        agg_50 = expvsflat_50
        agg_84 = expvsflat_84
        agg_lo = expvsflat_errlo
        agg_hi = expvsflat_errhi        

    if counter == 4:
        this_label = r'\textbf{Data vs. Exponential}'
        this_tab_tag = 'realvsexp'
        plot_bbox = True
        
    if counter == 5:
        this_label = r'\textbf{Data vs. Exponential}'
        this_tab_tag = 'realvsexp'
        plot_aggregate = True

        agg_16 = realvsexp_16
        agg_50 = realvsexp_50
        agg_84 = realvsexp_84
        agg_lo = realvsexp_errlo
        agg_hi = realvsexp_errhi
        plot_main_legend=True

    counter += 1
    
    axes[iax].plot([xmin,xmax],[0.,0.],color='black',linestyle='dashed',zorder=7)
    
    for this_source in corr_dict.keys():
        if corr_dict[this_source]['nclouds'] < nthresh:
            continue
            
        this_tab = corr_dict[this_source][this_tab_tag]
        
        this_x = np.log10(this_tab['xgeo'])
        this_y = np.log10(this_tab['twopt'])

        axes[iax].scatter(this_x,this_y,marker='o',s=10,
                          edgecolor='darkgray',facecolor='lightgray',
                          zorder=2)
        
        axes[iax].plot(this_x,this_y,zorder=1,linestyle='solid',color='gray')
        
    axes[iax].fill_betweenx(np.array([ymin,ymax]),xmin+np.array([0.,0.]),
                            (beam_scale)+np.array([0.,0.]),
                            hatch='////',facecolor='lightgray',alpha=0.15,
                            zorder=8,edgecolor=r'black',label=r'Beam')
        
    if plot_aggregate:
        agg_x = bins_xgeo            
        axes[iax].fill_between(np.log10(agg_x),np.log10(agg_16),np.log10(agg_84),
                               color='salmon',zorder=3,alpha=0.5, label=r'16-84\% range')
        agg_yerrlo = agg_50 - agg_lo
        agg_yerrhi = agg_50 + agg_hi
        axes[iax].fill_between(np.log10(agg_x),np.log10(agg_yerrlo),np.log10(agg_yerrhi),
                               color='cornflowerblue',zorder=4,alpha=0.65, label=r'Unc. on med.')
        
        axes[iax].plot(np.log10(agg_x),np.log10(agg_50),linestyle='solid',color='black',
                       zorder=5, linewidth=3, label=r'Median')
    
    bbox_props = dict(boxstyle="round", fc="lightgray", ec='black',alpha=0.9)
    yval = 0.95    

    if plot_bbox:
        ax.text(0.975, yval, this_label,
                ha='right', va='top',
                transform=ax.transAxes,
                size='small', bbox=bbox_props,
                zorder=5)
    
    if plot_main_legend:
        axes[iax].legend(fancybox=True, framealpha=0.95, loc='upper right', 
            ncol=1, fontsize=8, handlelength=2)    
        
#fig.tight_layout()
plt.savefig(png_name, bbox_inches='tight')

In [ ]:
png_name = '../plots/twopt_onepanel.png'

npanels = 1
ncol = 1
nrow = 1
ticklength = 5

xmin = +1.75
xmax = +4.0
ymin = 0.0
ymax = 2.5
xticks = np.arange(xmin,xmax,0.5)
yticks = np.arange(ymin,ymax,0.5)

mpl_setup('paper-1/1', aspect=3./6., fs=12)

fig, axes = plt.subplots(
    nrow, ncol, sharex='all', sharey='all',
    gridspec_kw={'wspace': 0.1, 'hspace': 0.1})

xlabel = r'$\log_{10}$ Separation [pc]'
ylabel = r'$1 + \omega$'

fig.text(0.015, 0.5, ylabel, va='center', ha='center', rotation='vertical', fontsize='large')
fig.text(0.5, 0.015, xlabel, va='center', ha='center', rotation='horizontal', fontsize='large')

rows_done = 0
counter = 0

beam_scale = np.log10(150.)

axes.set_xlim([xmin,xmax])
axes.set_xscale('linear')
axes.set_ylim([ymin,ymax])
axes.set_xticks(xticks)
axes.set_yticks(yticks)
axes.tick_params(direction='inout', length=ticklength)

axes.grid(True, linestyle='dotted',linewidth=0.5,color='black')

plot_main_legend = True
plot_bbox = True
plot_aggregate = True

this_label = r'\textbf{Data vs. Exponential}'
this_tab_tag = 'realvsexp'

agg_16 = realvsexp_16
agg_50 = realvsexp_50
agg_84 = realvsexp_84
agg_lo = realvsexp_errlo
agg_hi = realvsexp_errhi

axes.plot([xmin,xmax],[1.,1.],color='black',linestyle='dashed',zorder=7)

for this_source in corr_dict.keys():
    if corr_dict[this_source]['nclouds'] < nthresh:
        continue

    this_tab = corr_dict[this_source][this_tab_tag]

    this_x = np.log10(this_tab['xgeo'])
    this_y = (this_tab['twopt'])

    axes.scatter(this_x,this_y,marker='o',s=20,
                      edgecolor='darkgray',facecolor='lightgray',
                      zorder=2)

    axes.plot(this_x,this_y,zorder=1,linestyle='solid',color='gray')

axes.fill_betweenx(np.array([ymin,ymax]),xmin+np.array([0.,0.]),
                        (beam_scale)+np.array([0.,0.]),
                        hatch='////',facecolor='lightgray',alpha=0.25,
                        zorder=8,edgecolor=r'black',label=r'Beam')

if plot_aggregate:
    agg_x = bins_xgeo            
    axes.fill_between(np.log10(agg_x),(agg_16),(agg_84),
                           color='salmon',zorder=3,alpha=0.5, label=r'16-84\% range')
    agg_yerrlo = agg_50 - agg_lo
    agg_yerrhi = agg_50 + agg_hi
    axes.fill_between(np.log10(agg_x),(agg_yerrlo),(agg_yerrhi),
                           color='cornflowerblue',zorder=4,alpha=0.65, label=r'Unc. on med.')

    axes.plot(np.log10(agg_x),(agg_50),linestyle='solid',color='black',
                   zorder=5, linewidth=3, label=r'Median')

bbox_props = dict(boxstyle="round", fc="lightgray", ec='black',alpha=0.9)
axes.text(0.975, 0.01, this_label,
        ha='right', va='bottom',
        transform=ax.transAxes,
        size='medium', bbox=bbox_props,
        zorder=10)

if plot_main_legend:
    axes.legend(fancybox=True, framealpha=0.95, loc=(0.645,0.65), #loc='upper right', 
        ncol=1, fontsize=10, handlelength=2)    

#fig.tight_layout()
plt.savefig(png_name, bbox_inches='tight')